In [4]:
  # =========================
  # Synthetic ASR dataset generation via TTS
  # Генерирует 100 mp3-файлов по текстовым календарным запросам
  # =========================

  !pip -q install edge-tts nest_asyncio

  import json
  import shutil
  import asyncio
  import nest_asyncio
  from pathlib import Path
  from datetime import datetime

  import edge_tts

  nest_asyncio.apply()


  SAVE_DIR = Path("/content/inassist_synthetic_asr_dataset")
  AUDIO_DIR = SAVE_DIR / "audio"
  MANIFEST_PATH = SAVE_DIR / "manifest.json"
  ZIP_PATH = "/content/inassist_synthetic_asr_dataset"

  AUDIO_DIR.mkdir(parents=True, exist_ok=True)

  VOICE = "ru-RU-DmitryNeural"


  RATE = "+0%"
  VOLUME = "+0%"
  PITCH = "+0Hz"

  PROMPTS = [
    "Поставь встречу с куратором завтра в четырнадцать часов на один час.",
    "Добавь звонок с командой в пятницу в одиннадцать часов.",
    "Запланируй тренировку сегодня вечером на полтора часа.",
    "Создай событие разбор курсовой на понедельник в шестнадцать часов.",
    "Поставь напоминание про оплату интернета на двадцать пятое мая.",
    "Добавь встречу с преподавателем во вторник после обеда.",
    "Запиши консультацию по проекту на завтра в восемнадцать часов.",
    "Создай лекцию по математике в среду с десяти до одиннадцати часов.",
    "Поставь дедлайн по отчёту на воскресенье вечером.",
    "Добавь телефонный звонок с рекрутером на четверг в пятнадцать часов.",

    "Перенеси встречу с аналитиком с завтра на пятницу.",
    "Сдвинь звонок с командой на один час позже.",
    "Перенеси тренировку с вечера на утро.",
    "Измени встречу с куратором и поставь её на пятнадцать часов.",
    "Передвинь консультацию по проекту на следующий понедельник.",
    "Сдвинь разбор курсовой на полчаса вперёд.",
    "Перенеси встречу с преподавателем на следующую неделю.",
    "Измени длительность звонка на сорок пять минут.",
    "Поменяй время лекции на двенадцать часов.",
    "Перенеси дедлайн по отчёту на день раньше.",

    "Удали встречу с куратором завтра.",
    "Убери из календаря тренировку на сегодня.",
    "Удали событие разбор курсовой.",
    "Отмени звонок с командой в пятницу.",
    "Удали консультацию по проекту на следующей неделе.",
    "Убери дедлайн по отчёту из календаря.",
    "Отмени встречу с рекрутером в четверг.",
    "Удали все события с названием тестовая встреча.",
    "Убери лекцию по математике в среду.",
    "Отмени встречу с преподавателем во вторник.",

    "Найди свободное окно завтра на один час.",
    "Подбери свободное время в пятницу после пятнадцати часов.",
    "Найди свободный слот на полтора часа на следующей неделе.",
    "Когда у меня есть свободные два часа в понедельник?",
    "Подбери время для тренировки сегодня вечером.",
    "Найди свободное окно утром в среду.",
    "Есть ли у меня свободное время завтра после обеда?",
    "Найди ближайший свободный слот на тридцать минут.",
    "Подбери окно для звонка с командой на этой неделе.",
    "Найди свободное время завтра между двенадцатью и восемнадцатью часами.",

    "Что у меня запланировано на завтра?",
    "Покажи события на эту неделю.",
    "Какие встречи у меня сегодня после обеда?",
    "Покажи расписание на понедельник.",
    "Что у меня в календаре в пятницу?",
    "Есть ли у меня встречи завтра утром?",
    "Покажи все события с преподавателем.",
    "Найди в календаре встречу с куратором.",
    "Какие дедлайны у меня на этой неделе?",
    "Покажи ближайшие три события.",

    "Добавь встречу с дизайнером, когда будет свободное окно завтра.",
    "Запланируй звонок с командой на ближайший свободный час.",
    "Поставь консультацию по проекту на свободное время после обеда.",
    "Найди окно на полтора часа и поставь туда тренировку.",
    "Запланируй разбор отчёта в первый свободный слот в пятницу.",
    "Добавь встречу с аналитиком, когда у меня нет других дел.",
    "Поставь подготовку к зачёту на ближайшие свободные два часа.",
    "Найди свободное время вечером и добавь туда чтение статьи.",
    "Запланируй задачу дописать документацию на свободный слот завтра.",
    "Поставь звонок с рекрутером в ближайшее свободное окно на этой неделе.",

    "Перенеси завтрашнюю встречу с куратором на свободное время в пятницу.",
    "Сдвинь тренировку на ближайший свободный вечер.",
    "Перенеси консультацию по проекту туда, где будет окно на один час.",
    "Найди другое время для звонка с командой на этой неделе.",
    "Переставь лекцию на свободное окно после обеда.",
    "Перенеси встречу с преподавателем на ближайший свободный слот.",
    "Сдвинь разбор курсовой на день, когда у меня меньше встреч.",
    "Перенеси дедлайн по отчёту на ближайший рабочий день.",
    "Найди свободный слот завтра и перенеси туда встречу с рекрутером.",
    "Перенеси тренировку с утра на любое свободное время вечером.",

    "Добавь встречу завтра.",
    "Поставь звонок после обеда.",
    "Запланируй работу над проектом на вечер.",
    "Добавь событие с преподавателем.",
    "Перенеси встречу на более позднее время.",
    "Сдвинь звонок на другое время.",
    "Удали завтрашнее событие.",
    "Найди свободное окно на вечер.",
    "Поставь задачу на следующую неделю.",
    "Добавь встречу, когда будет удобно.",

    "Поставь подготовку презентации на завтра в тринадцать часов на два часа.",
    "Добавь проверку экспериментов сегодня в двадцать один час на один час.",
    "Запланируй обсуждение данных на среду в семнадцать часов.",
    "Поставь переписывание раздела отчёта на субботу утром.",
    "Добавь обновление резюме на воскресенье в пятнадцать часов.",
    "Создай событие тестирование сервера завтра с двенадцати до четырнадцати часов.",
    "Поставь разбор ошибок модели на пятницу вечером.",
    "Добавь звонок по практике на понедельник в одиннадцать часов.",
    "Запланируй подготовку к собеседованию каждый день в девятнадцать часов.",
    "Поставь написание руководства пользователя на ближайший свободный слот.",

    "Что у меня завтра между десятью и восемнадцатью часами?",
    "Есть ли у меня свободное окно перед встречей с куратором?",
    "Найди событие, где в названии есть слово курсовая.",
    "Покажи все встречи с командой за эту неделю.",
    "Найди ближайший телефонный звонок.",
    "Когда у меня следующая консультация?",
    "Покажи все события на выходных.",
    "Есть ли пересечения в расписании завтра?",
    "Покажи расписание на следующую среду.",
    "Составь краткую сводку моего календаря на неделю."
]
  # ---------- ГЕНЕРАЦИЯ ----------

  async def synthesize_one(index, text):
      filename = f"{index + 1:03d}.mp3"
      audio_path = AUDIO_DIR / filename

      communicate = edge_tts.Communicate(
          text=text,
          voice=VOICE,
          rate=RATE,
          volume=VOLUME,
          pitch=PITCH
      )

      await communicate.save(str(audio_path))

      return {
          "index": index + 1,
          "audio_file": filename,
          "text": text
      }


  async def main():
      items = []

      for i, text in enumerate(PROMPTS):
          print(f"[{i + 1}/{len(PROMPTS)}] Генерирую: {text}")
          item = await synthesize_one(i, text)
          items.append(item)

      manifest = {
          "dataset_name": "inassist_synthetic_asr_dataset",
          "created_at": datetime.now().isoformat(timespec="seconds"),
          "tts": {
              "engine": "edge-tts",
              "voice": VOICE,
              "rate": RATE,
              "volume": VOLUME,
              "pitch": PITCH
          },
          "items": items
      }

      with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
          json.dump(manifest, f, ensure_ascii=False, indent=2)

      zip_file = shutil.make_archive(ZIP_PATH, "zip", SAVE_DIR)

      print("\nГотово!")
      print(f"Аудио: {AUDIO_DIR}")
      print(f"Manifest: {MANIFEST_PATH}")
      print(f"Архив: {zip_file}")

      return zip_file


  zip_file = asyncio.get_event_loop().run_until_complete(main())

  from google.colab import files
  files.download(zip_file)

[1/100] Генерирую: Поставь встречу с куратором завтра в четырнадцать часов на один час.
[2/100] Генерирую: Добавь звонок с командой в пятницу в одиннадцать часов.
[3/100] Генерирую: Запланируй тренировку сегодня вечером на полтора часа.
[4/100] Генерирую: Создай событие разбор курсовой на понедельник в шестнадцать часов.
[5/100] Генерирую: Поставь напоминание про оплату интернета на двадцать пятое мая.
[6/100] Генерирую: Добавь встречу с преподавателем во вторник после обеда.
[7/100] Генерирую: Запиши консультацию по проекту на завтра в восемнадцать часов.
[8/100] Генерирую: Создай лекцию по математике в среду с десяти до одиннадцати часов.
[9/100] Генерирую: Поставь дедлайн по отчёту на воскресенье вечером.
[10/100] Генерирую: Добавь телефонный звонок с рекрутером на четверг в пятнадцать часов.
[11/100] Генерирую: Перенеси встречу с аналитиком с завтра на пятницу.
[12/100] Генерирую: Сдвинь звонок с командой на один час позже.
[13/100] Генерирую: Перенеси тренировку с вечера на утро.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>